In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

FILES = {
    'CO':  '../data/ekstraksi_fitur_co.csv',
    'NO2': '../data/ekstraksi_fitur_no2.csv',
    'SO2': '../data/ekstraksi_fitur_so2.csv',
}

def preprocess(df):
    feat_cols = [c for c in df.columns if c not in ['id', 'nama', 'daerah']]
    X = df[feat_cols].copy().fillna(df[feat_cols].median(numeric_only=True))
    const_cols = X.nunique()[X.nunique() <= 1].index.tolist()
    if const_cols:
        X = X.drop(columns=const_cols)
    Xs = StandardScaler().fit_transform(X)
    return Xs

for pol, path in FILES.items():
    df = pd.read_csv(path)
    Xs = preprocess(df)
    n_comp = min(37, Xs.shape[0], Xs.shape[1])
    Xp = PCA(n_components=n_comp, random_state=42).fit_transform(Xs)
    labels = KMeans(n_clusters=2, n_init=10, random_state=42).fit_predict(Xp)
    df['cluster'] = labels

    my_row = df[df['daerah'].str.contains('Sreseh|Sampang|Bundah', case=False, na=False)]
    my_cluster = my_row['cluster'].values[0]
    my_pos = df.index.get_loc(my_row.index[0])
    df['dist'] = [np.linalg.norm(Xp[i] - Xp[my_pos]) for i in range(len(df))]

    print(f'\n{"="*65}')
    print(f'POLUTAN {pol} — Kamu ({my_row["nama"].values[0]}) masuk Cluster {my_cluster}')
    print(f'{"="*65}')
    same = df[df['cluster'] == my_cluster].sort_values('dist')
    print(f'Anggota Cluster {my_cluster} (dari yang paling mirip ke kamu):')
    for _, r in same.iterrows():
        tag = ' ← KAMU' if 'Sreseh' in str(r['daerah']) or 'Sampang' in str(r['daerah']) else ''
        print(f"  [{r['cluster']}] id={r['id']:>2}  {r['nama']:<35} | {r['daerah']:<30} jarak={r['dist']:.3f}{tag}")
    other = df[df['cluster'] != my_cluster].sort_values('dist')
    print(f'\nAnggota Cluster {1-my_cluster} (kelompok lain):')
    for _, r in other.iterrows():
        print(f"  [{r['cluster']}] id={r['id']:>2}  {r['nama']:<35} | {r['daerah']}")
